# 01 数据准备（Data Prep）

本 Notebook 负责从 Google Drive / 本地数据源中加载原始采集数据，通过 **Error Simulator** 扰动正确样本生成训练对，输出为 `.npz` 供后续训练使用。

**流程**：
1. 挂载 Google Drive（Colab）
2. 加载 good rep 序列（.npy：`[T, 33, 3]` world 或 body-frame）
3. 转 body-frame + 归一化（若为 world）
4. 按 phase 重采样到 100 帧（可选）
5. **Error Simulator**：对每帧 Y* 扰动 → X̃，构造 `(X̃, Y_target)` 对
   - 含 **髋部上漂**（`hip_drift_prob`）模拟 MediaPipe「腰在胸」现象
6. 保存 `.npz`：`landmarks`, `phase`, `corrected`, `visibility`, `confidence`


In [ ]:
# 环境与路径（Colab 中请修改 PROJECT_ROOT 和 DATA_ROOT）
from pathlib import Path
import sys
import numpy as np

PROJECT_ROOT = Path("/content/ai_form_1")  # Colab clone 路径
DATA_ROOT = Path("/content/drive/MyDrive/ai_form_coach_data")  # 数据目录
if not Path("/content/drive").exists():  # 未挂载 Drive 时用本地路径
  DATA_ROOT = PROJECT_ROOT / "tools" / "train_denoiser" / "data"
OUT_NPZ = DATA_ROOT / "train_data.npz"

sys.path.insert(0, str(PROJECT_ROOT / "tools" / "train_denoiser" / "src"))

In [ ]:
# Colab: 挂载 Google Drive
try:
  from google.colab import drive
  drive.mount("/content/drive")
except ImportError:
  print("非 Colab 环境，跳过 Drive 挂载")

In [ ]:
# 加载 good rep 序列（.npy 格式：每个文件 shape [T, 33, 3]）
# 若为 world 坐标，需转 body-frame；若已是 body-frame 可跳过转换
from body_frame import to_body_frame

good_reps_dir = DATA_ROOT / "good_reps"
good_rep_paths = list(good_reps_dir.glob("*.npy")) if good_reps_dir.exists() else []
if not good_rep_paths:
  # 示例：创建一条虚拟 rep 用于测试
  np.random.seed(42)
  dummy = np.random.randn(50, 33, 3).astype(np.float32) * 0.1
  all_reps = [dummy]
else:
  all_reps = [np.load(str(p)).astype(np.float32) for p in good_rep_paths]

# 转为 body-frame（若输入为 world）
USE_WORLD_INPUT = True  # 若已是 body-frame 则设为 False
if USE_WORLD_INPUT:
  bf_reps = []
  for rep in all_reps:
    out = []
    vis = np.ones((rep.shape[0], 33), dtype=np.float32)
    for t in range(rep.shape[0]):
      bf, _ = to_body_frame(rep[t], vis[t])
      out.append(bf)
    bf_reps.append(np.stack(out))
else:
  bf_reps = all_reps

print(f"加载 {len(bf_reps)} 个 rep，长度: {[r.shape[0] for r in bf_reps]}")

In [ ]:
# 重采样到 100 帧（对齐 phase bins）
from scipy.interpolate import interp1d

TARGET_FRAMES = 100

def resample_rep(rep: np.ndarray, target: int = TARGET_FRAMES) -> np.ndarray:
  T = rep.shape[0]
  if T == target:
    return rep
  x_old = np.linspace(0, 1, T)
  x_new = np.linspace(0, 1, target)
  out = np.zeros((target, 33, 3), dtype=np.float32)
  for j in range(33):
    for c in range(3):
      f = interp1d(x_old, rep[:, j, c], kind="linear", fill_value="extrapolate")
      out[:, j, c] = f(x_new)
  return out

resampled = [resample_rep(r) for r in bf_reps]
phase_per_frame = np.linspace(0, 1, TARGET_FRAMES)

In [ ]:
# Error Simulator：从正确样本构造训练对 (X̃, Y_target)
from error_simulator import build_training_pairs

all_landmarks, all_phase, all_corrected, all_visibility, all_confidence = [], [], [], [], []
rng = np.random.default_rng(42)

for rep in resampled:
  phase_seq = np.linspace(0, 1, len(rep), dtype=np.float32)
  lm, ph, corr, vis, conf = build_training_pairs(
    rep,
    phase_seq,
    augment_per_frame=3,
    target_mode="same",  # 或 "next" 做一步前瞻
    rng=rng,
    shrug_prob=0.1,
    torso_lean_prob=0.1,
    amplitude_scale_prob=0.15,
    asymmetry_prob=0.1,
    hip_drift_prob=0.15,
    hip_drift_scale_range=(0.3, 0.8),
    jitter_std=0.02,
    dropout_prob=0.02,
  )
  all_landmarks.append(lm)
  all_phase.append(ph)
  all_corrected.append(corr)
  all_visibility.append(vis)
  all_confidence.append(conf)

landmarks = np.concatenate(all_landmarks, axis=0)
phase = np.concatenate(all_phase, axis=0)
corrected = np.concatenate(all_corrected, axis=0)
visibility = np.concatenate(all_visibility, axis=0)
confidence = np.concatenate(all_confidence, axis=0)

print(f"生成样本数: {len(landmarks)}")

In [ ]:
# 保存为 .npz
DATA_ROOT.mkdir(parents=True, exist_ok=True)
np.savez(
  OUT_NPZ,
  landmarks=landmarks,
  phase=phase,
  corrected=corrected,
  visibility=visibility,
  confidence=confidence,
)
print(f"已保存: {OUT_NPZ}")

In [ ]:
# 可视化：随机抽一帧，对比 用户(扰动后) vs 目标(正确)
from plotting import plot_skeleton_3d

idx = np.random.randint(len(landmarks))
fig = plot_skeleton_3d(
  landmarks[idx],
  corrected_points=corrected[idx],
  title=f"样本 #{idx} | phase={phase[idx].item():.2f}",
)
fig.show()